In [1]:
import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
import gzip
import pickle
import imageio
import tempfile   
import os
from IPython import display
from io import BytesIO

#### **FULLY CONNECTED NEURAL NETWORKS**

Every neuron from a layer is connected to every neuron from the next layer.
For each neuron: ther basic building blocks are a **linear operation** implemented through a matrix multiplication, and the result is used on a **non-linear element-wise activation function** to compute the result from the next layer.


The first step is to visualize function spaces by activation functions


In [3]:
def linear_link(x):
    return x
    
class FCLayer(nn.Module):
    def __init__(self, dim_in, dim_out, act):
        super().__init__()
        ## create parameters
        self.linear = nn.Linear(dim_in, dim_out)
        self.act = act
        
    def forward(self, x):
        fx = self.linear(x)
        return fx


class FCDNN(nn.Module):
    def __init__(self, dim_in, dim_out, neurons_hidden : list, hidden_activations:list, dropout_hidden, batch_norm: bool, add_residual:bool, link_function, loss_function):
        super().__init__()

        module_list = nn.ModuleList([])

        # input layer hidden layers
        for num_neur, act, drop in zip(neurons_hidden, hidden_activations, dropout_hidden):
            module_list.append(FCLayer(dim_in, num_neur, act, drop, batch_norm, add_residual))
            dim_in = num_neur

        # output layer
        o_layer = FCLayer(dim_in, dim_out, act = linear_link, drop = 0.0, batch_norm = False, add_residual = False)
        module_list.append(o_layer)

        self.layers = module_list
       
        ## Loss and link function
        self.link = link_function
        self.loss = loss_function
        
    def forward(self,x, apply_link):
        for l in self.layers:
            x = l(x)
        y = x
        if apply_link:
            y = self.link(y)
        
        return y

    def compute_loss(self,t,y):
        return self.loss(t,y)

Once we have created the basic structure of the network, we will try different activation functions and visualize the the different spaces of each of them. 
- We will study **(sigmoid, relu and tanh)** with different number of hidden layers and different number of neurons per layer.



In [7]:
#Task 1.2 

## function domain
N_min = -10
N_max = 10
N_points = 100

x_range = torch.linspace(N_min, N_max, N_points).reshape(N_points, 1)

## num functions to display
N_fun = 10

## Lets parameterize different models
activations = {
                'sigmoid' : torch.sigmoid, 
                'relu' : torch.relu, 
                'tanh' : torch.tanh
              }
num_hidden_layers = [1, 2, 3]
neurons_per_layer = [2, 10, 1024]

## create figure grid
fig, axes = plt.subplots(len(activations), len(num_hidden_layers) * len(neurons_per_layer), figsize = (200,80))

for i_a, (act_n, act_p) in enumerate(activations.items()):
    ax_acts = axes[i_a]
    
    counter_ax = 0
    for i_h, num_h in enumerate(num_hidden_layers):
        for i_n, n_l in enumerate(neurons_per_layer):
            ax_neur = ax_acts[counter_ax]
            
            for n_fun in range(N_fun):
                model = FCDNN(
                              dim_in = 1, 
                              dim_out = 1, 
                              neurons_hidden = [n_l]*num_h, 
                              hidden_activations = [act_p]*num_h,
                              link_function = linear_link, 
                              loss_function = None
                             )

                with torch.no_grad():
                    y_range = model.forward(x_range, apply_link = True)

                    ax_neur.plot(x_range, y_range, color = f'C{n_fun}', linewidth = 15)
            
            if counter_ax == 0:
                ax_neur.set_ylabel(f'{act_n}', fontsize = 80)
            
            ax_neur.set_title(f'hidden layers {num_h} neurons : {n_l}', fontsize = 80)
            ax_neur.tick_params(axis='both', labelsize=60)
            
            counter_ax += 1

TypeError: FCDNN.__init__() missing 3 required positional arguments: 'dropout_hidden', 'batch_norm', and 'add_residual'